# Preference Conditions Comparison

Cross-condition analysis for the 4 experimental conditions:
- **SFT baseline** — trained on annotated intents, no preference tuning
- **DPO-hard** — DPO on hard negatives from annotated data
- **DPO-judge** — DPO with extra rejecteds from LLM intent judge (bad_match taxonomy)
- **DPO-aug** — DPO on WildGuardMix-augmented data + augmented SFT init

Evaluated on 3 datasets: Annotated Intents (test), WildGuardTest, XSTest.

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

# ── Prediction directories ────────────────────────────────────────────────────
PRED_DIRS = {
    "SFT":       "../../data/predictions/sft_baseline",
    "DPO-hard":  "../../trained_models/causal/dpo-hard/predictions",
    "DPO-judge": "../../trained_models/causal/dpo-judge/predictions",
    "DPO-aug":   "/scratch/s4351495/intent-gen-trained_models/causal/dpo-augmented/predictions",
}

DATASETS = [
    ("Annotated Intents", "test_predictions.jsonl"),
    ("WildGuardTest",     "wildguardtest_predictions.jsonl"),
    ("XSTest",           "xstest_predictions.jsonl"),
]

CONDITIONS = list(PRED_DIRS.keys())

In [ ]:
# ── Load all predictions ──────────────────────────────────────────────────────
def load_jsonl(path):
    records = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records


preds = {}  # preds[condition][dataset_label] = list[dict]
for cond, pred_dir in PRED_DIRS.items():
    preds[cond] = {}
    for ds_label, filename in DATASETS:
        path = Path(pred_dir) / filename
        if path.exists() and path.stat().st_size > 0:
            preds[cond][ds_label] = load_jsonl(path)
            print(f"[OK]   {cond:12s} / {ds_label:20s}  ({len(preds[cond][ds_label])} records)")
        else:
            preds[cond][ds_label] = []
            print(f"[MISS] {cond:12s} / {ds_label}")

In [ ]:
# ── Harm metrics ─────────────────────────────────────────────────────────────
def harm_metrics(records):
    valid = [
        (r["true_harm"], r["predicted_harm"])
        for r in records
        if r.get("true_harm") in ("harmful", "safe")
        and r.get("predicted_harm") in ("harmful", "safe")
    ]
    if not valid:
        return {}
    y_true = [t for t, _ in valid]
    y_pred = [p for _, p in valid]
    labels = ["harmful", "safe"]
    return {
        "n":            len(valid),
        "n_unparsed":   len(records) - len(valid),
        "accuracy":     accuracy_score(y_true, y_pred),
        "f1_macro":     f1_score(y_true, y_pred, average="macro", labels=labels, zero_division=0),
        "f1_harmful":   f1_score(y_true, y_pred, pos_label="harmful", average="binary", zero_division=0),
        "f1_safe":      f1_score(y_true, y_pred, pos_label="safe",    average="binary", zero_division=0),
        "prec_harmful": precision_score(y_true, y_pred, pos_label="harmful", average="binary", zero_division=0),
        "rec_harmful":  recall_score(y_true, y_pred,    pos_label="harmful", average="binary", zero_division=0),
    }


# Build metrics dict: metrics[dataset][condition]
metrics = {}
for ds_label, _ in DATASETS:
    metrics[ds_label] = {}
    for cond in CONDITIONS:
        metrics[ds_label][cond] = harm_metrics(preds[cond][ds_label])

In [ ]:
# ── Harm metrics table ────────────────────────────────────────────────────────
METRIC_COLS = ["accuracy", "f1_macro", "f1_harmful", "rec_harmful", "f1_safe", "n_unparsed"]

for ds_label, _ in DATASETS:
    rows = []
    for cond in CONDITIONS:
        m = metrics[ds_label][cond]
        row = {"condition": cond}
        for col in METRIC_COLS:
            row[col] = round(m[col], 4) if col in m else None
        rows.append(row)
    df = pd.DataFrame(rows).set_index("condition")

    # Bold best per column (excluding n_unparsed)
    print(f"\n{'='*60}")
    print(f"  {ds_label}")
    print(f"{'='*60}")
    display(df.style.highlight_max(
        subset=[c for c in METRIC_COLS if c != "n_unparsed"],
        axis=0, props="font-weight: bold"
    ).format("{:.4f}", subset=[c for c in METRIC_COLS if c != "n_unparsed"]))

In [ ]:
# ── Intent quality metrics (Annotated Intents only) ───────────────────────────
# gold_intent is only populated in the annotated intents test split.

import evaluate as hf_evaluate
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

rouge = hf_evaluate.load("rouge")
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

DS_ANNOT = "Annotated Intents"

intent_rows = []
for cond in CONDITIONS:
    records = preds[cond][DS_ANNOT]
    pairs = [
        (r["gold_intent"], r["generated_intent"])
        for r in records
        if r.get("gold_intent") and r.get("generated_intent")
    ]
    if not pairs:
        intent_rows.append({"condition": cond, "n": 0, "rougeL": None, "sem_sim": None})
        continue

    refs  = [p[0] for p in pairs]
    hyps  = [p[1] for p in pairs]

    rouge_scores = rouge.compute(predictions=hyps, references=refs, use_aggregator=False)
    rougeL = float(np.mean(rouge_scores["rougeL"]))

    ref_emb  = embedder.encode(refs, convert_to_numpy=True, show_progress_bar=False)
    hyp_emb  = embedder.encode(hyps, convert_to_numpy=True, show_progress_bar=False)
    sims = [float(cosine_similarity([ref_emb[i]], [hyp_emb[i]])[0, 0]) for i in range(len(refs))]
    sem_sim = float(np.mean(sims))

    intent_rows.append({"condition": cond, "n": len(pairs), "rougeL": round(rougeL, 4), "sem_sim": round(sem_sim, 4)})

df_intent = pd.DataFrame(intent_rows).set_index("condition")
print("\nIntent quality (Annotated Intents test set)")
display(df_intent.style.highlight_max(subset=["rougeL", "sem_sim"], axis=0, props="font-weight: bold"))

In [ ]:
# ── Precompute intent embeddings for error analysis ───────────────────────────
# Compute per-record gold vs generated intent cosine similarity for all conditions.

annotated_records = {cond: preds[cond][DS_ANNOT] for cond in CONDITIONS}

intent_sims = {}  # intent_sims[cond] = list of (record, sim)

for cond in CONDITIONS:
    records = annotated_records[cond]
    valid = [
        r for r in records
        if r.get("gold_intent") and r.get("generated_intent")
        and r.get("true_harm") in ("harmful", "safe")
        and r.get("predicted_harm") in ("harmful", "safe")
    ]
    if not valid:
        intent_sims[cond] = []
        continue

    gold_emb = embedder.encode([r["gold_intent"] for r in valid], convert_to_numpy=True, show_progress_bar=False)
    gen_emb  = embedder.encode([r["generated_intent"] for r in valid], convert_to_numpy=True, show_progress_bar=False)
    sims = [float(cosine_similarity([gold_emb[i]], [gen_emb[i]])[0, 0]) for i in range(len(valid))]

    intent_sims[cond] = list(zip(valid, sims))
    print(f"{cond}: {len(valid)} records with intent pairs computed")

In [ ]:
# ── Error analysis: "accidentally correct" ────────────────────────────────────
# Label correct (predicted_harm == true_harm) but intent semantically diverges.
# These are cases where the model got the right answer for the wrong reason.

def accidentally_correct_df(cond, top_n=20):
    """Records where harm label is correct but intent similarity is lowest."""
    cases = [
        (r, sim) for r, sim in intent_sims[cond]
        if r["predicted_harm"] == r["true_harm"]
    ]
    cases.sort(key=lambda x: x[1])  # ascending sim = most divergent first
    rows = []
    for r, sim in cases[:top_n]:
        rows.append({
            "sim":             round(sim, 3),
            "true_harm":       r["true_harm"],
            "prompt":          r["prompt"][:120],
            "gold_intent":     r["gold_intent"][:200],
            "generated_intent": r["generated_intent"][:200],
        })
    return pd.DataFrame(rows)


for cond in CONDITIONS:
    df = accidentally_correct_df(cond, top_n=20)
    print(f"\n{'='*60}")
    print(f"  Accidentally correct — {cond} (lowest intent sim, label correct)")
    print(f"{'='*60}")
    if df.empty:
        print("  (no records)")
    else:
        display(df)

In [ ]:
# ── Error analysis: intent error causes label error ───────────────────────────
# Label wrong (predicted_harm != true_harm), sorted by intent divergence.
# These show where getting the intent wrong leads directly to a wrong harm label.

def label_wrong_df(cond, top_n=20):
    cases = [
        (r, sim) for r, sim in intent_sims[cond]
        if r["predicted_harm"] != r["true_harm"]
    ]
    cases.sort(key=lambda x: x[1])  # most divergent intent first
    rows = []
    for r, sim in cases[:top_n]:
        rows.append({
            "sim":              round(sim, 3),
            "true_harm":        r["true_harm"],
            "predicted_harm":   r["predicted_harm"],
            "prompt":           r["prompt"][:120],
            "gold_intent":      r["gold_intent"][:200],
            "generated_intent": r["generated_intent"][:200],
        })
    return pd.DataFrame(rows)


for cond in CONDITIONS:
    df = label_wrong_df(cond, top_n=20)
    print(f"\n{'='*60}")
    print(f"  Label wrong — {cond} (lowest intent sim, label incorrect)")
    print(f"{'='*60}")
    if df.empty:
        print("  (no records)")
    else:
        display(df)

In [ ]:
# ── Per-condition error breakdown ─────────────────────────────────────────────
# For each condition, count:
#   A) Label correct + intent high-sim (>= 0.7)  — label right, intent right
#   B) Label correct + intent low-sim  (< 0.7)   — accidentally correct
#   C) Label wrong                                — outright errors

SIM_THRESHOLD = 0.7

breakdown_rows = []
for cond in CONDITIONS:
    label_right_intent_right = 0
    label_right_intent_wrong = 0
    label_wrong = 0

    for r, sim in intent_sims[cond]:
        if r["predicted_harm"] == r["true_harm"]:
            if sim >= SIM_THRESHOLD:
                label_right_intent_right += 1
            else:
                label_right_intent_wrong += 1
        else:
            label_wrong += 1

    n = len(intent_sims[cond])
    breakdown_rows.append({
        "condition":                  cond,
        "n":                          n,
        "label_right_intent_right":   label_right_intent_right,
        "label_right_intent_wrong":   label_right_intent_wrong,
        "label_wrong":                label_wrong,
        "pct_accid_correct":          round(100 * label_right_intent_wrong / n, 1) if n else 0,
        "pct_label_wrong":            round(100 * label_wrong / n, 1) if n else 0,
    })

df_breakdown = pd.DataFrame(breakdown_rows).set_index("condition")
print(f"\nPer-condition breakdown  (intent sim threshold = {SIM_THRESHOLD})")
display(df_breakdown)